# Hyperspectral Image Classification with CNN and Transformer

**Author:** Alessandro Carotenuto  
**Student ID:** 1847282  
**Dataset:** Pavia University  

## Project aim

The project aims to classify land-cover pixels in a hyperspectral image.
A spatial-spectral patch centered on each labeled pixel is used as input.
A CNN learns local features, while a lightweight Transformer models non-local context.

## Setup 

### Paths

In [1]:
from pathlib import Path
import numpy as np

project_root = Path.cwd().parent
data_dir = project_root / "data" / "raw"

cube_path = data_dir / "PaviaU.mat"
ground_truth_path = data_dir / "PaviaU_gt.mat"

print("Cube:", cube_path)
print("Cube exists:", cube_path.exists())

print("Ground truth:", ground_truth_path)
print("Ground truth exists:", ground_truth_path.exists())

Cube: c:\Users\alex1\Documents\Cloned Repositories\Hyperspectral-Neural-Networks-Project\data\raw\PaviaU.mat
Cube exists: True
Ground truth: c:\Users\alex1\Documents\Cloned Repositories\Hyperspectral-Neural-Networks-Project\data\raw\PaviaU_gt.mat
Ground truth exists: True


### Loading MATLAB Files

In [2]:
from scipy.io import loadmat

cube_file = loadmat(cube_path)
ground_truth_file = loadmat(ground_truth_path)

print(cube_file.keys())
print(ground_truth_file.keys())

dict_keys(['__header__', '__version__', '__globals__', 'paviaU'])
dict_keys(['__header__', '__version__', '__globals__', 'paviaU_gt'])


### Dimension Check

In [3]:
cube = cube_file["paviaU"]
ground_truth = ground_truth_file["paviaU_gt"]

print("Cube:", cube.shape, cube.dtype)
print("Ground truth:", ground_truth.shape, ground_truth.dtype)
print("Cube range:", cube.min(), cube.max())
print("Labels:", np.unique(ground_truth))

Cube: (610, 340, 103) uint16
Ground truth: (610, 340) uint8
Cube range: 0 8000
Labels: [0 1 2 3 4 5 6 7 8 9]


### Class Count

In [4]:
class_names = {
    0: "Unlabeled",
    1: "Asphalt",
    2: "Meadows",
    3: "Gravel",
    4: "Trees",
    5: "Painted metal sheets",
    6: "Bare Soil",
    7: "Bitumen",
    8: "Self-Blocking Bricks",
    9: "Shadows",
}

labels, counts = np.unique(ground_truth, return_counts=True)

for label, count in zip(labels, counts):
    print(label, class_names[label], count)

0 Unlabeled 164624
1 Asphalt 6631
2 Meadows 18649
3 Gravel 2099
4 Trees 3064
5 Painted metal sheets 1345
6 Bare Soil 5029
7 Bitumen 1330
8 Self-Blocking Bricks 3682
9 Shadows 947


### Patch Configuration

In [5]:
from enum import IntEnum


class PatchSize(IntEnum):
    SMALLEST = 7
    SMALL = 11
    PAPER = 15
    LARGE = 19
    LARGEST = 23


PATCH_SIZE = PatchSize.PAPER
PATCH_RADIUS = int(PATCH_SIZE) // 2

print("Patch size:", int(PATCH_SIZE))
print("Patch radius:", PATCH_RADIUS)

Patch size: 15
Patch radius: 7


## Dataset Exploration

### Spectral Bands and Ground Truth

In [ ]:
import matplotlib.pyplot as plt


bands_to_show = [10, 50, 90]

fig, axes = plt.subplots(1, 4, figsize=(18, 5))

for axis, band_index in zip(axes[:3], bands_to_show):
    axis.imshow(cube[:, :, band_index], cmap="gray")
    axis.set_title(f"Band {band_index}")
    axis.axis("off")

ground_truth_image = axes[3].imshow(
    ground_truth,
    cmap="tab10",
    vmin=0,
    vmax=9,
)

axes[3].set_title("Ground Truth")
axes[3].axis("off")

plt.colorbar(ground_truth_image, ax=axes[3], fraction=0.046)
plt.tight_layout()
plt.show()

### Labeled Class Distribution

In [ ]:
labeled_mask = labels != 0
labeled_labels = labels[labeled_mask]
labeled_counts = counts[labeled_mask]
labeled_names = [class_names[label] for label in labeled_labels]

fig, axis = plt.subplots(figsize=(10, 6))
bars = axis.barh(labeled_names, labeled_counts, color="steelblue")

axis.set_title("Pavia University Labeled Class Distribution")
axis.set_xlabel("Number of labeled pixels")
axis.set_ylabel("Class")
axis.invert_yaxis()
axis.bar_label(bars, padding=4)

plt.tight_layout()
plt.show()

### Mean Spectral Signatures

In [ ]:
num_bands = cube.shape[2]
band_indices = np.arange(1, num_bands + 1)

fig, axis = plt.subplots(figsize=(12, 7))

for label in range(1, 10):
    class_pixels = cube[ground_truth == label]
    mean_signature = class_pixels.mean(axis=0)

    axis.plot(
        band_indices,
        mean_signature,
        label=class_names[label],
    )

axis.set_title("Mean Spectral Signature per Class")
axis.set_xlabel("Spectral band")
axis.set_ylabel("Mean pixel value")
axis.grid(alpha=0.25)
axis.legend(bbox_to_anchor=(1.02, 1), loc="upper left")

plt.tight_layout()
plt.show()